# Solutions · 17 · From a messy rheometer file to a report

Worked solutions to the exercises of [notebook 17](../notebooks/17_rheometer_file_to_report.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import io
from engrheo import datasets, fitting
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
lines = open(datasets.path("ketchup_rheometer_export.csv"), encoding="utf-8").read().splitlines()

def parse_export(lines):
    tests, i = {}, 0
    while i < len(lines):
        if lines[i].startswith("Test;"):
            name = lines[i].split(";")[1]; i += 1
            while not lines[i].startswith("Point;"):
                i += 1
            cols = [f"{h} ({u})" if u else h for h, u in zip(lines[i].split(";"), lines[i + 1].split(";"))]
            j = i + 2
            while j < len(lines) and lines[j].strip():
                j += 1
            tests[name] = pd.read_csv(io.StringIO("\n".join(lines[i + 2:j])), sep=";", decimal=",", header=None,
                                      names=cols, na_values=["---"])
            i = j
        else:
            i += 1
    return tests

tests = parse_export(lines)
up = tests["Flow curve (up)"].sort_values("Shear rate (1/s)")

## Exercise 1

Refit the upward flow curve *including* the two points flagged 'steady state not reached'. How much do the yield stress and its confidence interval change? Why is excluding them justified - and why must the exclusion be reported?

In [3]:
ok = up["Status"] == "ok"
for label, d in (("valid points only", up[ok]), ("including 'steady state not reached'", up)):
    f = fitting.fit_flow_curve(d["Shear rate (1/s)"], d["Shear stress (Pa)"], "herschel_bulkley")
    lo, hi = f.conf_int()["tau_y"]
    print(f"{label:<38}: tau_y = {f.params['tau_y']:5.2f} Pa (95 % CI {lo:.2f}-{hi:.2f}), scatter {100*f.sigma:.1f} %")

valid points only                     : tau_y = 14.95 Pa (95 % CI 14.25-15.65), scatter 2.0 %
including 'steady state not reached'  : tau_y = 18.65 Pa (95 % CI 16.18-21.11), scatter 13.0 %


The two unreliable points pull the fitted yield stress up and inflate the scatter - and they sit exactly where the
yield stress is determined, at the lowest rates. Excluding them is justified because the instrument itself
reported that the stress had not settled; reporting the exclusion lets a reader judge that decision, and
repeat the analysis either way.

## Exercise 2

The ketchup must flow from a squeezed bottle but not run off a plate. Which of the three yield-stress estimates is most relevant to each requirement? Compare all three with the true value (15 Pa).

In [4]:
print("true yield stress 15 Pa; estimates: HB up about 15 Pa, HB down about 13.5 Pa, flow point about 5 Pa (notebook 17)")

true yield stress 15 Pa; estimates: HB up about 15 Pa, HB down about 13.5 Pa, flow point about 5 Pa (notebook 17)


**Flowing from a squeezed bottle** is about the stress needed to make an undisturbed, rested product flow: the
upward-curve Herschel-Bulkley yield stress (≈ 15 Pa, matching the true value) is the relevant one. **Not running off a
plate** concerns a product that has just been sheared by squeezing and must hold its shape under gravity: the
lower, post-shear value from the downward curve is the conservative choice. The flow-point stress (≈ 5 Pa)
describes where the gel network first gives way in oscillation - it is lower because yielding in oscillation
starts at small strains, long before steady flow; it is the most sensitive indicator of structural change
between batches, not a design value for flow.

## Exercise 3

**Implement it yourself:** turn the parsing and the checks into one reusable function `load_export(path)` that returns the tests as DataFrames with SI units (Pa·s instead of mPa·s, strain as a fraction) and a list of excluded points, and test it on the file.

In [5]:
def load_export(path):
    raw = open(path, encoding="utf-8").read().splitlines()
    out, excluded = {}, []
    for name, d in parse_export(raw).items():
        d = d.copy()
        if "Viscosity (mPa·s)" in d:
            d["Viscosity (Pa·s)"] = d.pop("Viscosity (mPa·s)") / 1000
        if "Strain (%)" in d:
            d["Strain (-)"] = d.pop("Strain (%)") / 100
        bad = (d["Status"] != "ok") | d.isna().any(axis=1)
        excluded += [f"{name}, point {int(p)}: {s}" for p, s in zip(d.loc[bad, "Point"], d.loc[bad, "Status"])]
        out[name] = d[~bad].reset_index(drop=True)
    return out, excluded

data, excluded = load_export(datasets.path("ketchup_rheometer_export.csv"))
print({k: len(v) for k, v in data.items()}); print("\n".join(excluded))

{'Amplitude sweep': 21, 'Frequency sweep': 16, 'Flow curve (up)': 17, 'Flow curve (down)': 18}
Flow curve (up), point 1: Steady state not reached
Flow curve (up), point 2: Steady state not reached
Flow curve (down), point 7: Measurement failed


One function now turns any export of this format into clean SI tables plus an explicit list of excluded points -
the basis for a report that can be regenerated for every new batch.